# BTCPredictor2 - GPU Training

## Before running
1. Enable GPU: **Runtime -> Change runtime type -> A100 GPU** (recommended)
2. Run Cell 1 (clones repo + installs packages)
3. Run Cell 2 (mounts Drive and links yearly feature files)
4. Run Cells 3, 4, 5 to train TFT, BiLSTM, and Meta

## What to upload to Google Drive
Upload the entire `data/yearly/` folder from your local BTCPredictor2 to your Drive.
It contains: `2022_merged.csv`, `2023_merged.csv`, `2024_merged.csv`, `2025_merged.csv`, `2026_merged.csv`

**Estimated time on A100 GPU:**
- TFT: 2-4 hours
- BiLSTM: 30-60 minutes
- Meta: 5 minutes

In [ ]:
# ============================================================
# CELL 1 - SETUP
# Always deletes and re-clones so you get the latest GitHub push.
# ============================================================

GITHUB_URL = 'https://github.com/chefo919/BTCPredictor2.git'

import shutil, os
if os.path.exists('/content/BTCPredictor2'):
    shutil.rmtree('/content/BTCPredictor2')
    print('Removed old clone.')

!git clone {GITHUB_URL} /content/BTCPredictor2
!cd /content/BTCPredictor2 && git log --oneline -3

import sys
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

!pip install -q xgboost joblib ta scikit-learn-intelex

os.makedirs('/content/BTCPredictor2/data/yearly', exist_ok=True)
os.makedirs('/content/BTCPredictor2/models/saved', exist_ok=True)

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f'GPU: {gpus[0].name}  |  Mixed precision: ON')
else:
    print('WARNING: No GPU — Runtime > Change runtime type > GPU')

import config
config.BATCH_TFT    = 512
config.BATCH_BILSTM = 1024

from features.engineer import get_feature_groups
from models import tft_model
groups = get_feature_groups()

days = config.SEQ_LEN_TFT * tft_model.DOWNSAMPLE // (60 * 24)
print()
print(f'BiLSTM features: {len(groups["bilstm"])}  (1m/15m/30m — {config.SEQ_LEN_BILSTM}min minute-resolution context)')
print(f'TFT    features: {len(groups["tft_dynamic"])}  (h1/h4/d1 — {config.SEQ_LEN_TFT} steps x {tft_model.DOWNSAMPLE}min = {days} days context)')
print()
print(f'TFT:    SEQ={config.SEQ_LEN_TFT} x {tft_model.DOWNSAMPLE}min/step | HORIZON={config.HORIZON_TFT}min | batch={config.BATCH_TFT}')
print(f'BiLSTM: SEQ={config.SEQ_LEN_BILSTM} x 1min/step  | HORIZON={config.HORIZON_BILSTM}min | batch={config.BATCH_BILSTM}')
print()
print('Done. Run Cell 2 to link the training data.')

In [ ]:
# ============================================================
# CELL 2 - MOUNT DRIVE AND LINK YEARLY DATA FILES
# Upload your local data/yearly/ folder to Google Drive first.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DEST_DIR = '/content/BTCPredictor2/data/yearly'
os.makedirs(DEST_DIR, exist_ok=True)

# Look for yearly files in common Drive locations
YEARLY_FILES = ['2022_merged.csv', '2023_merged.csv', '2024_merged.csv',
                '2025_merged.csv', '2026_merged.csv']

search_dirs = [
    '/content/drive/MyDrive/BTCPredictor2/data/yearly',
    '/content/drive/MyDrive/BTCPredictor2/yearly',
    '/content/drive/MyDrive/yearly',
    '/content/drive/MyDrive',
]

found_dir = None
for d in search_dirs:
    if os.path.exists(d) and any(os.path.exists(os.path.join(d, f)) for f in YEARLY_FILES):
        found_dir = d
        break

if found_dir is None:
    print('ERROR: Could not find yearly CSV files in Drive.')
    print('Please upload your local data/yearly/ folder to Google Drive.')
    print('Checked locations:')
    for d in search_dirs:
        print(f'  {d}')
else:
    linked = []
    for fname in YEARLY_FILES:
        src = os.path.join(found_dir, fname)
        dst = os.path.join(DEST_DIR, fname)
        if os.path.exists(src):
            if not os.path.exists(dst):
                os.symlink(src, dst)
            sz = os.path.getsize(src) / 1024**2
            import pandas as _pd
            rc = sum(1 for _ in open(src)) - 1
            print(f'  {fname}: {rc:,} rows  ({sz:.0f} MB)')
            linked.append(fname)
        else:
            print(f'  {fname}: NOT FOUND (skipping)')
    print()
    print(f'Linked {len(linked)} yearly files from {found_dir}')
    print('Ready. Run Cell 3 to train TFT.')

In [ ]:
# ============================================================
# CELL 3 - TRAIN TFT ENSEMBLE  (estimated: 1-2h on A100 for 3 seeds)
# D_MODEL=64 (was 32) -- 4x more parameters, same memory footprint.
# 3 seeds trained with different random inits, predictions averaged at inference.
# batch=512 safe: attention [512,4,168,168] = 55 MB float16.
# ============================================================

import os, sys, time, json
import numpy as np
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')

import config
config.BATCH_TFT = 512

import pandas as pd
from features.engineer import get_feature_groups
from models import tft_model

groups           = get_feature_groups()
TFT_DYN_FEATURES = groups['tft_dynamic']
TFT_STA_FEATURES = groups['tft_static']

days = config.SEQ_LEN_TFT * tft_model.DOWNSAMPLE // (60 * 24)
print(f'TFT features:  {len(TFT_DYN_FEATURES)}  (h1/h4/d1)')
print(f'TFT sequence:  {config.SEQ_LEN_TFT} steps x {tft_model.DOWNSAMPLE}min = {days} days')
print(f'D_MODEL:       {config.D_MODEL}  (was 32)')
print(f'Ensemble seeds:{tft_model.N_ENSEMBLE}')
print()

YEARLY_DIR  = 'data/yearly'
START_DATE  = '2022-02-01'
CUTOFF_DATE = config.TRAINING_CUTOFF_DATE
start_ts    = pd.Timestamp(START_DATE,  tz='UTC')
cutoff_ts   = pd.Timestamp(CUTOFF_DATE, tz='UTC')

dfs = []
for fname in sorted(os.listdir(YEARLY_DIR)):
    if not fname.endswith('_merged.csv'):
        continue
    df_y = pd.read_csv(os.path.join(YEARLY_DIR, fname), parse_dates=['time'])
    if df_y['time'].dt.tz is None:
        df_y['time'] = pd.to_datetime(df_y['time'], utc=True)
    df_y = df_y[(df_y['time'] >= start_ts) & (df_y['time'] <= cutoff_ts)]
    if not df_y.empty:
        dfs.append(df_y)
df = pd.concat(dfs, ignore_index=True).sort_values('time').reset_index(drop=True)
print(f'Rows (1m): {len(df):,}  ->  ~{len(df)//tft_model.DOWNSAMPLE:,} hourly rows after downsampling')
print()

t0 = time.time()
results = tft_model.train_ensemble(df, TFT_DYN_FEATURES, TFT_STA_FEATURES)
elapsed = time.time() - t0

avg_test = float(np.mean([r['test_acc'] for r in results]))
avg_val  = float(np.mean([r['val_acc']  for r in results]))

# Save accuracies for Cell 5
acc_path = 'models/saved/model_accuracies.json'
existing = json.load(open(acc_path)) if os.path.exists(acc_path) else {}
existing.update({'tft': avg_test, 'tft_val': avg_val})
with open(acc_path, 'w') as f:
    json.dump(existing, f, indent=2)

print()
print('TFT ENSEMBLE COMPLETE')
for i, r in enumerate(results):
    print(f'  Seed {i}: test={r["test_acc"]:.4f}  val={r["val_acc"]:.4f}')
print(f'  Average:  test={avg_test:.4f}  val={avg_val:.4f}')
print(f'  Time:     {int(elapsed//3600)}h {int((elapsed%3600)//60)}m')
print()
print('Run Cell 4 to train BiLSTM ensemble.')

In [ ]:
# ============================================================
# CELL 4 - TRAIN BILSTM ENSEMBLE  (estimated: 1-2h on A100 for 3 seeds)
# 3 seeds with different random inits, predictions averaged at inference.
# ============================================================

import os, sys, time, json
import numpy as np
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')

import config
config.BATCH_BILSTM = 1024

import pandas as pd
from features.engineer import get_feature_groups
from models import bilstm_model

groups          = get_feature_groups()
BILSTM_FEATURES = groups['bilstm']

print(f'BiLSTM features:  {len(BILSTM_FEATURES)}  (1m/15m/30m)')
print(f'Ensemble seeds:   {bilstm_model.N_ENSEMBLE}')
print()

YEARLY_DIR  = 'data/yearly'
START_DATE  = '2022-02-01'
CUTOFF_DATE = config.TRAINING_CUTOFF_DATE
start_ts    = pd.Timestamp(START_DATE,  tz='UTC')
cutoff_ts   = pd.Timestamp(CUTOFF_DATE, tz='UTC')

dfs = []
for fname in sorted(os.listdir(YEARLY_DIR)):
    if not fname.endswith('_merged.csv'):
        continue
    df_y = pd.read_csv(os.path.join(YEARLY_DIR, fname), parse_dates=['time'])
    if df_y['time'].dt.tz is None:
        df_y['time'] = pd.to_datetime(df_y['time'], utc=True)
    df_y = df_y[(df_y['time'] >= start_ts) & (df_y['time'] <= cutoff_ts)]
    if not df_y.empty:
        dfs.append(df_y)
df = pd.concat(dfs, ignore_index=True).sort_values('time').reset_index(drop=True)
print(f'Rows: {len(df):,}  ({START_DATE} -> {CUTOFF_DATE})')
print()

t0 = time.time()
results = bilstm_model.train_ensemble(df, BILSTM_FEATURES)
elapsed = time.time() - t0

avg_test = float(np.mean([r['test_acc'] for r in results]))
avg_val  = float(np.mean([r['val_acc']  for r in results]))

# Update accuracies for Cell 5
acc_path = 'models/saved/model_accuracies.json'
existing = json.load(open(acc_path)) if os.path.exists(acc_path) else {}
existing.update({'bilstm': avg_test, 'bilstm_val': avg_val})
with open(acc_path, 'w') as f:
    json.dump(existing, f, indent=2)

print()
print('BiLSTM ENSEMBLE COMPLETE')
for i, r in enumerate(results):
    print(f'  Seed {i}: test={r["test_acc"]:.4f}  val={r["val_acc"]:.4f}')
print(f'  Average:  test={avg_test:.4f}  val={avg_val:.4f}')
print(f'  Time:     {int(elapsed//3600)}h {int((elapsed%3600)//60)}m')
print()
print('Run Cell 5 to train Meta and download models.')

In [ ]:
# ============================================================
# CELL 5 - TRAIN META (AGREEMENT-BASED ROUTING) + DOWNLOAD  (~5 minutes)
#
# The meta-learner answers: "given this market snapshot, which base model
# is more likely to be correct HERE?" — NOT just "predict price direction."
#
# Two routing paths:
#   AGREE    (|p_tft - p_bilstm| < 0.10):
#       Both models agree — static error-reciprocal blend, no gate needed.
#   DISAGREE (gap >= 0.10):
#       A correctness gate trained ONLY on these rows learns:
#           gate(market_snapshot, gap) → P(TFT_correct | disagreement)
#           final = gate_prob × p_tft + (1 - gate_prob) × p_bilstm
#
# Why better than standard stacking at ~50% base accuracy:
#   - Gate trains where signal is cleanest (one model demonstrably right).
#   - Dynamic per-sample trust vs fixed global weights.
#   - Learns regime sensitivity: trending → TFT wins; vol spike → BiLSTM wins.
#
# OOF timeline:
#   [0-70% train] [+24h gap] [70-90% OOF] [+24h gap] [90-100% test]
# 24h purge gaps prevent training labels from referencing OOF prices.
# ============================================================

import os, sys, time, shutil, json
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import numpy as np
import pandas as pd
import config
from features.engineer import get_feature_groups
from models import tft_model, bilstm_model, meta_model

groups           = get_feature_groups()
BILSTM_FEATURES  = groups['bilstm']
TFT_DYN_FEATURES = groups['tft_dynamic']
TFT_STA_FEATURES = groups['tft_static']
ALL_NEEDED       = BILSTM_FEATURES + TFT_DYN_FEATURES + TFT_STA_FEATURES

# Load data
YEARLY_DIR  = 'data/yearly'
START_DATE  = '2022-02-01'
CUTOFF_DATE = config.TRAINING_CUTOFF_DATE
start_ts    = pd.Timestamp(START_DATE,  tz='UTC')
cutoff_ts   = pd.Timestamp(CUTOFF_DATE, tz='UTC')

dfs = []
for fname in sorted(os.listdir(YEARLY_DIR)):
    if not fname.endswith('_merged.csv'):
        continue
    df_y = pd.read_csv(os.path.join(YEARLY_DIR, fname), parse_dates=['time'])
    if df_y['time'].dt.tz is None:
        df_y['time'] = pd.to_datetime(df_y['time'], utc=True)
    df_y = df_y[(df_y['time'] >= start_ts) & (df_y['time'] <= cutoff_ts)]
    if not df_y.empty:
        dfs.append(df_y)
df = pd.concat(dfs, ignore_index=True).sort_values('time').reset_index(drop=True)
df = df.dropna(subset=ALL_NEEDED).reset_index(drop=True)

# OOF split with 24-hour purge gaps
PURGE      = 24 * 60
n_total    = len(df)
train_end  = int(n_total * 0.70)
oof_start  = train_end  + PURGE
oof_end    = int(n_total * 0.90)
test_start = oof_end    + PURGE

df_oof  = df.iloc[oof_start:oof_end].copy()
df_test = df.iloc[test_start:].copy()

print(f'Total rows:    {n_total:,}')
print(f'Train:         0 - {train_end:,}  (70%)')
print(f'OOF (meta):    {oof_start:,} - {oof_end:,}  ({len(df_oof):,} rows, 24h purge each side)')
print(f'Test:          {test_start:,} - end  ({len(df_test):,} rows)')
print(f'Using ensemble: TFT={tft_model._ensemble_ready()}  BiLSTM={bilstm_model._ensemble_ready()}')
print()

val_X_dyn = df_oof[TFT_DYN_FEATURES].values.astype('float32')
val_X_sta = df_oof[TFT_STA_FEATURES].values.astype('float32')
val_X_bi  = df_oof[BILSTM_FEATURES].values.astype('float32')

acc_path       = 'models/saved/model_accuracies.json'
saved_acc      = json.load(open(acc_path)) if os.path.exists(acc_path) else {}
tft_val_err    = 1.0 - saved_acc.get('tft_val', 0.51)
bilstm_val_err = 1.0 - saved_acc.get('bilstm_val', 0.51)

print('Generating OOF predictions (ensemble averages automatically)...')
val_tft_probs    = tft_model.predict_proba_batch(val_X_dyn, val_X_sta)
val_bilstm_probs = bilstm_model.predict_proba_batch(val_X_bi)

print('Training agreement-based routing meta-learner...')
meta_results = meta_model.train(df_oof, val_tft_probs, val_bilstm_probs,
                                 tft_val_err, bilstm_val_err,
                                 TFT_DYN_FEATURES + TFT_STA_FEATURES)
w = meta_results.get('weights', {})

print()
print('=' * 50)
print('TRAINING COMPLETE')
print(f'  TFT accuracy:    {saved_acc.get("tft", 0):.3f}  (ensemble of {config.N_ENSEMBLE} seeds)')
print(f'  BiLSTM accuracy: {saved_acc.get("bilstm", 0):.3f}  (ensemble of {config.N_ENSEMBLE} seeds)')
print(f'  Meta routing acc:{meta_results["train_acc"]:.3f}  (OOF, agreement-based gate)')
print(f'  Gate CV acc:     {meta_results.get("gate_cv_acc", 0):.3f}  '
      f'({meta_results.get("n_disagree", 0):,} disagreement rows, '
      f'{meta_results.get("pct_disagree", 0):.1f}% of OOF)')
print(f'  Static fallback  TFT: {w.get("tft", 0):.3f}  BiLSTM: {w.get("bilstm", 0):.3f}')
print(f'  Data window:     {START_DATE} -> {CUTOFF_DATE}')
print('=' * 50)

# ── Zip all ensemble models + meta for download ────────────────────────────
OUT = '/content/models_output'
os.makedirs(OUT, exist_ok=True)

for s in range(config.N_ENSEMBLE):
    for fname in [f'tft_s{s}.keras', f'tft_scaler_s{s}.pkl',
                  f'bilstm_s{s}.keras', f'bilstm_scaler_s{s}.pkl']:
        src = f'models/saved/{fname}'
        if os.path.exists(src):
            shutil.copy(src, f'{OUT}/{fname}')

for fname in ['tft_n_static.txt', 'meta_xgb.pkl',
              'model_accuracies.json', 'training_cutoff.txt']:
    src = f'models/saved/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/{fname}')

saved = os.listdir(OUT)
print(f'\nPackaged {len(saved)} files: {sorted(saved)}')
shutil.make_archive('/content/btc_models', 'zip', OUT)

from google.colab import files
print()
print('Downloading btc_models.zip to your PC...')
files.download('/content/btc_models.zip')
print()
print('Extract the zip and copy all files into your local models/saved/ folder.')
print('Then run: python papertrading/backtest.py --start 2026-04-15')